In [1]:
import os, openai

api_key = os.environ.get("DEEPSEEK_API_KEY")
base_url = "https://api.deepseek.com/"
model = "deepseek-v4-flash"

## 标签生成
+ 流程: 传入一段非结构化文本(附带结构化描述) -> 使用语言模型生成结构化输出（分析输入文本），按照传入的结构描述创建响应
+ 要求：生成包含文本情感对象，添加文本语言的标签
+ 返回：包含情感标签和语言标签的对象

## 提取用例
+ 流程：从文本中提取特定实体，其中实体由结构化描述表示，而非通过语言模型分析文本；使用语言模型扫描文本提取元素列表

In [2]:
# 导入标准配置
from typing import List
from pydantic import BaseModel, Field
from langchain_classic.utils.openai_functions import convert_pydantic_to_openai_tool

In [3]:
class Tagging(BaseModel):
    # 告诉语言模型期望提取的数据结构
    """Tag the piece of text with particular info."""
    sentiment: str = Field(description="sentiment of text, should be `pos`, `neg`, `neutral`")
    # 记录文本的语言，并指定编码
    language: str = Field(description="language of text (should be ISO 639-1 code)")

In [4]:
convert_pydantic_to_openai_tool(Tagging)

{'type': 'function',
 'function': {'name': 'Tagging',
  'description': 'Tag the piece of text with particular info.',
  'parameters': {'properties': {'sentiment': {'description': 'sentiment of text, should be `pos`, `neg`, `neutral`',
     'type': 'string'},
    'language': {'description': 'language of text (should be ISO 639-1 code)',
     'type': 'string'}},
   'required': ['sentiment', 'language'],
   'type': 'object'}}}

In [5]:
from langchain_classic.prompts import ChatPromptTemplate
from langchain_openai.chat_models import ChatOpenAI

In [6]:
model = ChatOpenAI(
    model=model,
    api_key=api_key,
    base_url=base_url,
    temperature=0

)

In [7]:
tagging_functions = [convert_pydantic_to_openai_tool(Tagging)]

In [8]:
prompt = ChatPromptTemplate.from_messages([
    ("system", "Think carefully, and then tag the text as instructed"),
    ("user", "{input}")
])

In [9]:
model_with_function = model.bind_tools(
    tagging_functions,
    tool_choice="Tagging",
    extra_body={"thinking": {"type": "disabled"}}, # 需要关闭思考模式才能强制使用tool
)

In [10]:
tagging_chain = prompt | model_with_function

In [11]:
response = tagging_chain.invoke({"input": "I love langchain"})

In [12]:
response

AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 49, 'prompt_tokens': 337, 'total_tokens': 386, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 128, 'image_tokens': None, 'text_tokens': None}, 'prompt_cache_hit_tokens': 128, 'prompt_cache_miss_tokens': 209}, 'model_provider': 'openai', 'model_name': 'deepseek-flash', 'system_fingerprint': 'aeb56401ca74e127821c4f9126dcb669', 'id': 'd060f384-588e-4845-bcb3-e04f6e4992a5', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a09f23-fe2f-76c1-a581-82f88f3856d8-0', tool_calls=[{'name': 'Tagging', 'args': {'sentiment': 'pos', 'language': 'en'}, 'id': 'call_00_amH3YC1pgZvFgDJUW8Ha6075', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 337, 'output_tokens': 49, 'total_tokens': 386, 'input_token_details': {'cache_read': 128}, 'output_token_details': {}})

In [13]:
response = tagging_chain.invoke({"input": "non mi piace questo cibo"})

In [14]:
response.tool_calls

[{'name': 'Tagging',
  'args': {'sentiment': 'neg', 'language': 'it'},
  'id': 'call_00_ClZkyuiCbgHurkctEfHy6457',
  'type': 'tool_call'}]

In [15]:
# 将argument的值解析成json
from langchain_core.output_parsers.openai_tools import JsonOutputToolsParser

In [16]:
tagging_chain = prompt | model_with_function | JsonOutputToolsParser()

In [17]:
response = tagging_chain.invoke({"input": "non mi piace questo cibo"})

In [18]:
response[0]["args"]

{'sentiment': 'neg', 'language': 'it'}

In [19]:
# 标签提取
from typing import Optional

class Person(BaseModel):
    """Information about a person."""
    name: str = Field(description="person's name")
    age: Optional[int] = Field(description="person's age")

In [20]:
class Information(BaseModel):
    """Information to extract."""
    people: List[Person] = Field(description="List of info about people")

In [21]:
convert_pydantic_to_openai_tool(Information)

{'type': 'function',
 'function': {'name': 'Information',
  'description': 'Information to extract.',
  'parameters': {'properties': {'people': {'description': 'List of info about people',
     'items': {'description': 'Information about a person.',
      'properties': {'name': {'description': "person's name",
        'type': 'string'},
       'age': {'anyOf': [{'type': 'integer'}, {'type': 'null'}],
        'description': "person's age"}},
      'required': ['name', 'age'],
      'type': 'object'},
     'type': 'array'}},
   'required': ['people'],
   'type': 'object'}}}

In [22]:
extraction_functions = [convert_pydantic_to_openai_tool(Information)]
# extraction_model = model.bind_tools(extraction_functions)

In [23]:
# 写法2,此时写法有问题，需要修改
extraction_model = model.bind(
    tools=extraction_functions,
    tool_choice={
        "type": "function",
        "function": {
            "name" : "Information"
        }
    },
    extra_body={
        # 关闭思考模式
        "thinking": {"type": "disabled"}
    }
)

In [24]:
response = extraction_model.invoke("Joe is 30,his mom is Martha,,and Martha looks like 40")

In [25]:
# 当模型不知道人物的年龄时，会直接赋值为None
response.tool_calls[0]["args"]

{'people': [{'name': 'Joe', 'age': 30}, {'name': 'Martha', 'age': 40}]}

In [26]:
prompt = ChatPromptTemplate.from_messages([
    ("system","Extract the relevant information,if not explicitly provided do not guess.Extract partial info"),
    # 当限制了获取个人信息的条件时，语言模型对于looks like这类模糊话语不会直接推测，当只有出现确定话语例如is，才会提取出信息
    ("user","{input}")
])

In [27]:
# 创建新的提取链
extraction_chain = prompt | extraction_model

In [28]:
response = extraction_chain.invoke("Joe is 30,his mom is Martha,,and Martha is 40")


In [29]:
response.tool_calls[0]["args"]

{'people': [{'name': 'Joe', 'age': 30}, {'name': 'Martha', 'age': 40}]}

In [30]:
# 将AI输出进行解析
extraction_chain = prompt | extraction_model | JsonOutputToolsParser()

In [31]:
response = extraction_chain.invoke("Joe is 30,his mom is Martha,,and Martha is 40")

In [32]:
for p in response[0]["args"]["people"]:
    print(f"name:{p["name"]}\nage:{p["age"]}\n")

name:Joe
age:30

name:Martha
age:40



In [33]:
# 在原始输出当中存在多余的people字段，导入其他的OutputParser来处理当前的输出，以除去people字段
from langchain_classic.output_parsers import JsonOutputKeyToolsParser
# 从输出当中查找特殊key，并且对其进行提取

In [34]:
# 重新创建链
extraction_chain = (prompt
                    | extraction_model
                    | JsonOutputKeyToolsParser(key_name="Information", first_tool_only=True)
                    | (lambda r: r["people"])
                    )

In [35]:
response = extraction_chain.invoke("Joe is 30,his mom is Martha,,and Martha is 40")


In [36]:
response

[{'name': 'Joe', 'age': 30}, {'name': 'Martha', 'age': 40}]

In [41]:
# 加载文章
from langchain_classic.document_loaders import WebBaseLoader
loader = WebBaseLoader("https://lilianweng.github.io/posts/2023-06-23-agent")
documents = loader.load()

In [44]:
doc = documents[0]

In [45]:
page_content = doc.page_content[:10000] # 获取前一万个字符

In [46]:
print(page_content[1000:])

gent System Overview#
In a LLM-powered autonomous agent system, LLM functions as the agent’s brain, complemented by several key components:

Planning

Subgoal and decomposition: The agent breaks down large tasks into smaller, manageable subgoals, enabling efficient handling of complex tasks.
Reflection and refinement: The agent can do self-criticism and self-reflection over past actions, learn from mistakes and refine them for future steps, thereby improving the quality of final results.


Memory

Short-term memory: I would consider all the in-context learning (See Prompt Engineering) as utilizing short-term memory of the model to learn.
Long-term memory: This provides the agent with the capability to retain and recall (infinite) information over extended periods, often by leveraging an external vector store and fast retrieval.


Tool use

The agent learns to call external APIs for extra information that is missing from the model weights (often hard to change after pre-training), inclu

In [47]:
# 创建Pydantic类进行标签和提取操作
class Overview(BaseModel):
    """Overview of a section of text"""
    summary: str = Field(description="Provide a concise summary of the content")
    language: str = Field(description="Provide the language that the content is written in.")
    keywords: str = Field(description="Provide keywords related to the content")

In [55]:
overview_tagging_function = [
    convert_pydantic_to_openai_tool(Overview)
]

# 绑定工具
tagging_model = model.bind(
    tools = overview_tagging_function,
    tool_choice={
        "type": "function",
        "function": {
            "name": "Overview"
        }
    },
    extra_body={
        # 需要关闭思考模式
        "thinking" : {
            "type" : "disabled"
        }
    }
)

# 创建标签链，并且携带结构化输出
tagging_chain = prompt | tagging_model | JsonOutputToolsParser()

In [56]:
tagging_chain.invoke({"input": page_content})

[{'args': {'summary': 'The text is a blog post titled "LLM Powered Autonomous Agents" by Lilian Weng, dated June 23, 2023. It discusses building autonomous agents with large language models (LLMs) as the core controller, citing examples like AutoGPT, GPT-Engineer, and BabyAGI. The post outlines an agent system overview with three key components: Planning (task decomposition and self-reflection), Memory (short-term and long-term), and Tool Use. It then details planning techniques including Chain of Thought (CoT), Tree of Thoughts, LLM+P, ReAct, Reflexion, Chain of Hindsight (CoH), and Algorithm Distillation (AD). The text is cut off mid-sentence in the Algorithm Distillation section.',
   'language': 'English',
   'keywords': 'LLM, autonomous agents, planning, task decomposition, self-reflection, memory, tool use, Chain of Thought, Tree of Thoughts, ReAct, Reflexion, Chain of Hindsight, Algorithm Distillation'},
  'type': 'Overview'}]

In [57]:
# 提取文章当中提到的所有论文
class Paper(BaseModel):
    """Information about papers mentioned."""
    title: str
    author: Optional[str]

class Info(BaseModel):
    """Information to extract"""
    papers: List[Paper]

In [58]:
paper_extraction_function = [
    convert_pydantic_to_openai_tool(Info)
]

extraction_model = model.bind_tools(paper_extraction_function)

extraction_chain = prompt | extraction_model | JsonOutputToolsParser()

In [60]:
extraction_chain.invoke({"input":page_content})

[{'args': {'papers': [{'title': 'Chain of Thought (CoT)',
     'author': 'Wei et al. 2022'},
    {'title': 'Tree of Thoughts', 'author': 'Yao et al. 2023'},
    {'title': 'LLM+P', 'author': 'Liu et al. 2023'},
    {'title': 'ReAct', 'author': 'Yao et al. 2023'},
    {'title': 'Reflexion', 'author': 'Shinn & Labash 2023'},
    {'title': 'Chain of Hindsight (CoH)', 'author': 'Liu et al. 2023'},
    {'title': 'Algorithm Distillation (AD)', 'author': 'Laskin et al. 2023'},
    {'title': 'RL^2', 'author': 'Duan et al. 2017'}]},
  'type': 'Info'}]

In [65]:
template = """
A article will be passed to you. Extract from it all papers that are mentioned by this article.

Do not extract the name of the article itself. If no papers are mentioned that's fine - you don't need to extract any! Just return an empty list.

Do not make up or guess ANY extra information. Only extract what exactly is in the text.
"""
prompt = ChatPromptTemplate.from_messages([
     ("system",template),
     ("human","{input}")
 ])

In [97]:
# extraction_chain = prompt | extraction_model | JsonOutputToolsParser()
extraction_chain = prompt | extraction_model | JsonOutputKeyToolsParser(key_name="Info",first_tool_only=True) | (lambda r: (r or {}).get("papers", []))

In [98]:
response =  extraction_chain.invoke({"input":page_content})

In [99]:
response

[{'title': 'Chain of thought', 'author': 'Wei et al. 2022'},
 {'title': 'Tree of Thoughts', 'author': 'Yao et al. 2023'},
 {'title': 'LLM+P', 'author': 'Liu et al. 2023'},
 {'title': 'ReAct', 'author': 'Yao et al. 2023'},
 {'title': 'Reflexion', 'author': 'Shinn & Labash 2023'},
 {'title': 'Chain of Hindsight', 'author': 'Liu et al. 2023'},
 {'title': 'Algorithm Distillation', 'author': 'Laskin et al. 2023'},
 {'title': 'RL^2', 'author': 'Duan et al. 2017'}]

In [100]:
response = extraction_chain.invoke({"input":"hi!"})

In [101]:
response

[]

In [102]:
# 对全文进行处理，对文本进行切割
from langchain_classic.text_splitter import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(chunk_overlap=0)

In [106]:
# 使用递归进行处理
splits = text_splitter.split_text(doc.page_content)

In [107]:
len(splits)

15

In [108]:
splits

["LLM Powered Autonomous Agents | Lil'Log\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\nLil'Log\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n|\n\n\n\n\n\n\nPosts\n\n\n\n\nArchive\n\n\n\n\nSearch\n\n\n\n\nTags\n\n\n\n\nFAQ\n\n\n\n\n\n\n\n\n\n      LLM Powered Autonomous Agents\n    \nDate: June 23, 2023  |  Estimated Reading Time: 31 min  |  Author: Lilian Weng\n\n\n \n\n\nTable of Contents\n\n\n\nAgent System Overview\n\nComponent One: Planning\n\nTask Decomposition\n\nSelf-Reflection\n\n\nComponent Two: Memory\n\nTypes of Memory\n\nMaximum Inner Product Search (MIPS)\n\n\nComponent Three: Tool Use\n\nCase Studies\n\nScientific Discovery Agent\n\nGenerative Agents Simulation\n\nProof-of-Concept Examples\n\n\nChallenges\n\nCitation\n\nReferences\n\n\n\n\n\nBuilding agents with LLM (large language model) as its core controller is a cool concept. Several proof-of-concepts demos, such as AutoGPT, GPT-Engineer and BabyAGI, serve as inspiring examples. The potentiality o

In [112]:
# 创建一个合并列表的函数
def flatten(matrix):
    flat_list = []
    for row in matrix:
        flat_list += row
    return flat_list

In [113]:
flatten([[1,2],[3,4]])

[1, 2, 3, 4]

In [114]:
print(splits[0])

LLM Powered Autonomous Agents | Lil'Log






































Lil'Log

















|






Posts




Archive




Search




Tags




FAQ









      LLM Powered Autonomous Agents
    
Date: June 23, 2023  |  Estimated Reading Time: 31 min  |  Author: Lilian Weng


 


Table of Contents



Agent System Overview

Component One: Planning

Task Decomposition

Self-Reflection


Component Two: Memory

Types of Memory

Maximum Inner Product Search (MIPS)


Component Three: Tool Use

Case Studies

Scientific Discovery Agent

Generative Agents Simulation

Proof-of-Concept Examples


Challenges

Citation

References





Building agents with LLM (large language model) as its core controller is a cool concept. Several proof-of-concepts demos, such as AutoGPT, GPT-Engineer and BabyAGI, serve as inspiring examples. The potentiality of LLM extends beyond generating well-written copies, stories, essays and programs; it can be framed as a powerful general problem solver.
Agent S

In [115]:
from langchain_classic.schema.runnable import  RunnableLambda

In [117]:
prep = RunnableLambda(
    # 字典对应每一次分割的输入
    lambda x: [{"input": doc} for doc in text_splitter.split_text(x)]
)

In [118]:
prep.invoke("hi")

[{'input': 'hi'}]

In [119]:
chain = prep | extraction_chain.map() | flatten # 对文本进行回分割 -> 将分割的文本传递给链

In [120]:
chain.invoke(doc.page_content)

[{'title': 'Chain of thought', 'author': 'Wei et al. 2022'},
 {'title': 'Tree of Thoughts', 'author': 'Yao et al. 2023'},
 {'title': 'LLM+P', 'author': 'Liu et al. 2023'},
 {'title': 'ReAct', 'author': 'Yao et al. 2023'},
 {'title': 'Reflexion', 'author': 'Shinn & Labash 2023'},
 {'title': 'Chain of Hindsight', 'author': 'Liu et al. 2023'},
 {'title': 'Algorithm Distillation', 'author': 'Laskin et al. 2023'},
 {'title': 'Shinn & Labash, 2023', 'author': None},
 {'title': 'RL^2', 'author': 'Duan et al. 2017'},
 {'title': 'Laskin et al. 2023', 'author': 'Laskin et al.'},
 {'title': 'Miller 1956', 'author': 'Miller'},
 {'title': 'MRKL', 'author': 'Karpas et al. 2022'},
 {'title': 'TALM (Tool Augmented Language Models)',
  'author': 'Parisi et al. 2022'},
 {'title': 'Toolformer', 'author': 'Schick et al. 2023'},
 {'title': 'HuggingGPT', 'author': 'Shen et al. 2023'},
 {'title': 'ChemCrow', 'author': None},
 {'title': 'Boiko et al. (2023)', 'author': 'Boiko et al.'},
 {'title': 'Generative 